In [1]:
import xarray as xr, netCDF4 as nc, numpy as np, pandas as pd, os, datetime

import glob

from scipy.spatial import cKDTree

import matplotlib.pyplot as plt
import plotly.graph_objects as go

THIS CODE IS NO LONGER CORRECT OR ACTIVE

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

ehf_fpath = '/scratch/ng72/ms5578/hw_files'
nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/raw'
write_path = '/scratch/ng72/ms5578/time_series'
raw_NEM_cache = '/scratch/ng72/ms5578/nemosis_cache'

In [3]:
sdate, edate = "2009/10/01 00:00:00", "2019/03/31 23:59:59"

In [4]:
netcdf_files = glob.glob(os.path.join(ehf_fpath, "*.nc")) 

In [5]:
EHF_ds = xr.open_mfdataset('/scratch/ng72/ms5578/hw_files/HW_EHF_2016_2017.nc',
                         engine='netcdf4',
                         chunks='auto')
EHF_ds = EHF_ds.chunk(lon=-1,lat=-1,time="350Mb")

# EHF_ds = ds.sel(time=slice(sdate, edate))
EHF_ds

<xarray.Dataset> Size: 12GB
Dimensions:      (time: 365, lat: 646, lon: 1082)
Coordinates:
  * time         (time) datetime64[ns] 3kB 2016-07-01T12:00:00 ... 2017-06-30...
  * lat          (lat) float64 5kB -57.97 -57.86 -57.75 ... 12.76 12.87 12.98
  * lon          (lon) float64 9kB 88.48 88.59 88.7 88.81 ... 207.2 207.3 207.4
    height       float64 8B ...
    crs          int32 4B ...
Data variables:
    EHF_flag     (lat, lon, time) float64 2GB dask.array<chunksize=(646, 1082, 62), meta=np.ndarray>
    EHF_val      (time, lat, lon) float64 2GB dask.array<chunksize=(62, 646, 1082), meta=np.ndarray>
    HW_EHF_avg   (lat, lon, time) float64 2GB dask.array<chunksize=(646, 1082, 62), meta=np.ndarray>
    HW_EHF_peak  (lat, lon, time) float64 2GB dask.array<chunksize=(646, 1082, 62), meta=np.ndarray>
    tas_3d_avg   (lat, lon, time) float64 2GB dask.array<chunksize=(646, 1082, 62), meta=np.ndarray>
    tas_3d_peak  (lat, lon, time) float64 2GB dask.array<chunksize=(646, 1082, 62), meta=np.ndarray>

In [6]:
gen_df = pd.read_csv(f"{nmap_path}/nmap.csv")
gen_df.drop(gen_df.columns[[3, 1, 4, 5]], axis=1, inplace=True)

gen_df.columns = (
    gen_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^\w_]', '', regex=True)
    .str.replace('__', '_')
)
gen_df = gen_df.rename(columns={'duid': 'DUID'})

First we join the generator information with the EHF info, sticking to daily intervals

In [7]:
def merge_nearest(ds,df):
    # Extract latitude and longitude grids
    lat_grid = ds['lat'].values
    lon_grid = ds['lon'].values
    
    # Convert gen_df lat/lon to numpy arrays for vectorized operations
    gen_lats = df['lat'].values
    gen_lons = df['lon'].values
    
    # Find the nearest indices for all latitudes and longitudes in gen_df
    lat_indices = np.abs(lat_grid[:, None] - gen_lats).argmin(axis=0)
    lon_indices = np.abs(lon_grid[:, None] - gen_lons).argmin(axis=0)
    
    # Get the nearest latitudes and longitudes
    nearest_lats = lat_grid[lat_indices]
    nearest_lons = lon_grid[lon_indices]
    
    # Create DataArrays for vectorized selection
    nearest_lats_da = xr.DataArray(nearest_lats, dims="DUID")
    nearest_lons_da = xr.DataArray(nearest_lons, dims="DUID")
    
    new_ds = ds.sel(lat=nearest_lats_da, lon=nearest_lons_da)
    new_ds = new_ds.assign_coords(DUID=df['DUID'])

    return new_ds

Performing some other operations to tidy up heatwave_info

In [8]:
hw_info = merge_nearest(EHF_ds, gen_df).to_dataframe().reset_index()
hw_info['time'] = pd.to_datetime(hw_info['time'])
hw_info = hw_info.replace(1.000000e+20, np.nan)
hw_info = hw_info.drop(['height','crs'], axis=1)
hw_info = hw_info.sort_values(by=['DUID', 'time'])


Here we number consecutive heatwave days

In [9]:
def number_heatwave_days(group):
    # Identify where heatwave days start (either first row or break in sequence)
    is_heatwave = group['EHF_flag'] == 1
    # Create a group ID for each continuous block of heatwave days
    group['event_group'] = (is_heatwave & (is_heatwave != is_heatwave.shift(1))).cumsum()
    # Set non-heatwave days to NaN in group
    group.loc[~is_heatwave, 'event_group'] = pd.NA
    # Now, within each event_group, number the days
    group['HW_event_day'] = group.groupby('event_group').cumcount() + 1
    # Drop the helper column
    group = group.drop(columns='event_group')
    return group

# Apply by DUID
hw_info = hw_info.groupby('DUID', group_keys=False).apply(number_heatwave_days, include_groups=True)
hw_info = hw_info.drop(columns=['lat','lon'])

/jobfs/142481510.gadi-pbs/ipykernel_2241648/3713384521.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hw_info = hw_info.groupby('DUID', group_keys=False).apply(number_heatwave_days, include_groups=True)


In [10]:
hw_info.to_csv(f"{write_path}/gen_hw_status.csv",
              index=False)

In [12]:
hw_info['HW_event_day'].max()

25.0

In [13]:
hw_info.nlargest(20, 'HW_event_day')

,time,DUID,EHF_flag,EHF_val,HW_EHF_avg,HW_EHF_peak,tas_3d_avg,tas_3d_peak,HW_event_day
113301,2017-02-18 12:00:00,GANGARR1,1.0,1.014811,2.745009,13.471772,300.978814,301.627604,25.0
113344,2017-02-18 12:00:00,WANDBG1,1.0,1.014811,2.745009,13.471772,300.978814,301.627604,25.0
113464,2017-02-18 12:00:00,WANDBL1,1.0,1.014811,2.745009,13.471772,300.978814,301.627604,25.0
112814,2017-02-17 12:00:00,GANGARR1,1.0,0.749593,2.745009,13.471772,301.402832,301.627604,24.0
112857,2017-02-17 12:00:00,WANDBG1,1.0,0.749593,2.745009,13.471772,301.402832,301.627604,24.0
112977,2017-02-17 12:00:00,WANDBL1,1.0,0.749593,2.745009,13.471772,301.402832,301.627604,24.0
112327,2017-02-16 12:00:00,GANGARR1,1.0,0.605713,2.745009,13.471772,301.498454,301.914469,23.0
112370,2017-02-16 12:00:00,WANDBG1,1.0,0.605713,2.745009,13.471772,301.498454,301.914469,23.0
112490,2017-02-16 12:00:00,WANDBL1,1.0,0.605713,2.745009,13.471772,301.498454,301.914469,23.0
111840,2017-02-15 12:00:00,GANGARR1,1.0,1.301676,2.745009,13.471772,302.261773,303.652344,22.0
